In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# ---------------------------------------------------------
# FASE 1: Ingestión, Remapeo de Clases y Preprocesamiento
# ---------------------------------------------------------
train_df = pd.read_csv('../data/sign_mnist_train.csv')
test_df = pd.read_csv('../data/sign_mnist_test.csv')

# Extraer las 24 etiquetas únicas originales (0..24 sin el 9) y crear diccionarios de mapeo (0..23)
unique_labels = sorted(train_df['label'].unique())
label_to_idx = {orig_label: new_idx for new_idx, orig_label in enumerate(unique_labels)}
idx_to_letter = {new_idx: chr(65 + orig_label) for new_idx, orig_label in enumerate(unique_labels)}

# Mapear etiquetas al rango continuo [0, 23]
y_full = train_df['label'].map(label_to_idx).values
y_test = test_df['label'].map(label_to_idx).values

# Extraer píxeles, normalizar a [0, 1] y redimensionar a tensores 3D (28, 28, 1)
X_full = train_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = test_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Partición estratificada: 85% Entrenamiento (23,336), 15% Validación (4,119)
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.15, random_state=42, stratify=y_full
)

# Creación de pipelines de alto rendimiento con tf.data
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=5000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

W0000 00:00:1793298523.990033   14752 cpu_allocator_impl.cc:174] Allocation of 73181696 exceeds 10% of free system memory.


In [ ]:
# ---------------------------------------------------------
# FASE 2: Diseño del Grafo Computacional (CNN + MLP)
# ---------------------------------------------------------
def build_cnn_mlp_model(input_shape=(28, 28, 1), num_classes=24):
    inputs = layers.Input(shape=input_shape, name="input_image")
    
    # Aumento de datos en GPU (Sin volteo horizontal para preservar la geometría de la seña)
    x = layers.RandomRotation(factor=0.08, name="aug_rotation")(inputs)
    x = layers.RandomZoom(height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08), name="aug_zoom")(x)
    x = layers.RandomTranslation(height_factor=0.05, width_factor=0.05, name="aug_translation")(x)
    
    # --- BLOQUE EXTRACTOR ESPACIAL (CNN) ---
    # Bloque Convolucional 1: Captura bordes y contrastes básicos -> (14, 14, 32)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False, name="conv1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation('relu', name="relu1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)
    
    # Bloque Convolucional 2: Captura contornos de dedos y palma -> (7, 7, 64)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False, name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Activation('relu', name="relu2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)
    
    # Bloque Convolucional 3: Articulaciones y configuraciones complejas -> (7, 7, 128)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False, name="conv3_a")(x)
    x = layers.BatchNormalization(name="bn3_a")(x)
    x = layers.Activation('relu', name="relu3_a")(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False, name="conv3_b")(x)
    x = layers.BatchNormalization(name="bn3_b")(x)
    x = layers.Activation('relu', name="relu3_b")(x)
    
    # --- EL PUENTE DIMENSIONAL ---
    # Reduce (7, 7, 128) -> Vector 1D de (128,) evitando explosión de parámetros
    x = layers.GlobalAveragePooling2D(name="bridge_gap")(x)
    
    # --- BLOQUE CLASIFICADOR (MLP) ---
    x = layers.Dense(256, activation='relu', name="mlp_dense_1")(x)
    x = layers.BatchNormalization(name="mlp_bn_1")(x)
    x = layers.Dropout(0.4, name="mlp_dropout_1")(x)
    
    x = layers.Dense(128, activation='relu', name="mlp_dense_2")(x)
    x = layers.Dropout(0.3, name="mlp_dropout_2")(x)
    
    outputs = layers.Dense(num_classes, activation='softmax', name="mlp_output")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="SignMNIST_CNN_MLP")

model = build_cnn_mlp_model()
model.summary()


In [ ]:
# ---------------------------------------------------------
# FASE 3 y 4: Optimización (Adam) y Callbacks de Control
# ---------------------------------------------------------
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

training_callbacks = [
    # Detiene el entrenamiento si val_loss no mejora en 6 épocas y restaura el mejor modelo
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce el learning rate de Adam a la mitad si val_loss se estanca por 3 épocas
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Guarda en disco únicamente el mejor estado alcanzado
    callbacks.ModelCheckpoint(
        filepath='best_sign_mnist_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=training_callbacks
)


In [ ]:
# ---------------------------------------------------------
# FASE 5: Evaluación Aislada y Diagnóstico de Errores
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\n[EVALUACIÓN FINAL EN TEST] Loss: {test_loss:.4f} | Accuracy: {test_acc*100:.2f}%\n")

# Predicciones en el conjunto de prueba aislado
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Reporte detallado por letra del alfabeto (A-Y sin J)
target_names = [f"Letra {idx_to_letter[i]}" for i in range(24)]
print(classification_report(y_test, y_pred, target_names=target_names, digits=4))

# Visualización de Curvas y Matriz de Confusión
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curva de Pérdida
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
axes[0].set_title('Convergencia de la Función de Pérdida (Adam + EarlyStopping)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Sparse Categorical Crossentropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
letter_ticks = [idx_to_letter[i] for i in range(24)]
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=letter_ticks, yticklabels=letter_ticks, ax=axes[1])
axes[1].set_title('Matriz de Confusión en el Set de Prueba (7,172 imágenes)')
axes[1].set_xlabel('Predicción del Modelo')
axes[1].set_ylabel('Etiqueta Real')

plt.tight_layout()
plt.show()